In [ ]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# Failure Analysis\n",
    "\n",
    "Load test-ID and test-OOD datasets, plot the distribution of relative L2 errors per sample,\n",
    "and inspect the worst OOD failures. This notebook is meant to be run *after* training.\n",
    "\n",
    "To keep this file small, actual code is minimal; expand as needed."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import torch, numpy as np\n",
    "import matplotlib.pyplot as plt\n",
    "from src.data.dataset import SchrodingerDataset\n",
    "from src.models.fno import FNO1d\n",
    "from src.eval.metrics import rel_l2\n",
    "\n",
    "dev = 'cuda' if torch.cuda.is_available() else 'cpu'\n",
    "ckpt = torch.load('results/fno_single_seed42.pt', map_location=dev, weights_only=False)\n",
    "m = FNO1d(**{k: ckpt['cfg']['model'][k] for k in ('modes','width','n_layers')}).to(dev)\n",
    "m.load_state_dict(ckpt['model']); m.eval()"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "def per_sample_errors(path):\n",
    "    ds = SchrodingerDataset(path)\n",
    "    errs = []\n",
    "    with torch.no_grad():\n",
    "        for i in range(len(ds)):\n",
    "            inp, tgt = ds[i]\n",
    "            pred = m(inp.unsqueeze(0).to(dev)).cpu().squeeze(0)\n",
    "            errs.append(rel_l2(pred.unsqueeze(0), tgt.unsqueeze(0)).item())\n",
    "    return np.array(errs)\n",
    "\n",
    "e_id = per_sample_errors('data/test_id.pt')\n",
    "e_ood = per_sample_errors('data/test_ood.pt')\n",
    "print('ID   median/p95/max:', np.median(e_id), np.percentile(e_id, 95), e_id.max())\n",
    "print('OOD  median/p95/max:', np.median(e_ood), np.percentile(e_ood, 95), e_ood.max())"
   ]
  }
 ],
 "metadata": {"kernelspec": {"display_name": "Python 3", "language": "python", "name": "python3"}},
 "nbformat": 4,
 "nbformat_minor": 5
}

: 

# Schrödinger-FNO

A reproducible PyTorch project that trains a Fourier Neural Operator (FNO) as a
learned propagator for the 1D Schrödinger equation (ħ = m = 1):
